# Day 3, Notebook 2: the rows a profile cannot see

Notebook 1 profiled the columns. Every count came back explainable except one: `order_id` holds 49
distinct values across 50 rows.

This notebook is about that missing one, about the order at the far end of the amount column, and
about what you ship at the end of the day.

Position in the day:

`[profile the columns] > [decide per field] > **[find the hidden rows]** > [investigate the extremes] > [ship with the log]`

The map below is where this notebook sits in the day and what it adds. Every
notebook in the programme opens on the same pair, so you know where you are before you read a line.

The cell that draws it also brings in the programme's helper. `scripts/c2kit.py` is found by
walking up from this notebook's own folder, which is what lets the same file run whether you
pressed Run All here or a script ran it for you. The helper loads the day's data from `../data/`,
draws every diagram you see in these notebooks, and runs the checks that tell you a cell did what
it claimed.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["profiling the columns", "the rows a profile cannot see", "hands-on: the full pass", "hands-on: which dataset"], lit=1, title="the day's notebooks", show=False),
    kit.flow(["the dedupe that finds nothing", "the count that disagrees", "state an identity rule", "the order at the end of the column", "what ships"], title="what this notebook adds", show=False),
)

## Setup

Same file, same functions, one cell so this runs cold in a fresh Codespace.

In [2]:
import csv
import os

DATA_DIR = "../data"
ORDERS_CSV = f"{DATA_DIR}/C2_W01_D03_orders_STUDENT.csv"
COMPANION_CSV = f"{DATA_DIR}/C2_W01_D03_companion_STUDENT.csv"
OUTPUT_DIR = "output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(ORDERS_CSV) as f:
    orders = list(csv.DictReader(f))


def normalise_amount(raw):
    """Yesterday's function, unchanged."""
    return int(raw)


def clean_record(record):
    """Yesterday's function, unchanged."""
    keeper = dict(record)
    keeper["amount"] = normalise_amount(record["amount"])
    return keeper


def clean_records(rows):
    """Yesterday's function, unchanged."""
    clean, rejects = [], []
    for r in rows:
        try:
            clean.append(clean_record(r))
        except ValueError as e:
            rejects.append({"order_id": r["order_id"], "reason": str(e)})
    return clean, rejects

decisions = []

def record_decision(field, finding, choice, reason):
    decisions.append({"field": field, "finding": finding, "choice": choice, "reason": reason})

print(f"{len(orders)} orders loaded")

50 orders loaded


In [3]:
kit.flow(["the dedupe that finds nothing", "the count that disagrees", "state an identity rule", "the order at the end of the column", "what ships"], lit=0)

## Section 1: the duplicate check that finds nothing

The obvious way to look for duplicates is to ask whether any two rows are identical.

In [4]:
whole_rows = [tuple(r.values()) for r in orders]
duplicates = len(whole_rows) - len(set(whole_rows))
print(f"duplicate rows by whole-record comparison: {duplicates}")

duplicate rows by whole-record comparison: 0


In [5]:
kit.check("the whole-record check reports zero duplicates", duplicates == 0)
kit.check("and the file has fifty rows", len(orders) == 50)
kit.check("so nothing on this screen says anything is wrong", duplicates == 0,
          "which is what a wrong answer looks like")

Zero. Clean dataset, move on.

Nothing raised. Nothing was highlighted. This is what a wrong answer looks like on a good day.

In [6]:
kit.flow(["the dedupe that finds nothing", "the count that disagrees", "state an identity rule", "the order at the end of the column", "what ships"], lit=1)

## Section 2: the count that disagrees

Section 1 plus one new element: a second number to compare the first against.

In [7]:
ids = [r["order_id"] for r in orders]
print(f"rows: {len(ids)}")
print(f"distinct order_ids: {len(set(ids))}")
print(f"unexplained: {len(ids) - len(set(ids))}")

rows: 50
distinct order_ids: 49
unexplained: 1


In [8]:
kit.check("fifty rows", len(ids) == 50)
kit.check("forty-nine distinct order ids", len(set(ids)) == 49)
kit.check("two numbers on two lines that disagree, and no code noticed",
          len(ids) != len(set(ids)))

### The two numbers, and the gap between them

In [9]:
kit.side_by_side(
    kit.flow(["compare whole rows", "0 duplicates"], lit=1,
             title="what the dedupe saw", show=False),
    kit.flow(["count distinct ids", "49 across 50 rows"], lit=1,
             title="what the id count saw", show=False),
)

Two numbers, on two different lines, that disagree. Nothing in the code noticed, because nothing in
the code was asked to compare them.

That comparison is the whole of today's first lesson, and it costs one line.

In [10]:
from collections import Counter

repeated = [oid for oid, n in Counter(ids).items() if n > 1]
print("order_ids appearing more than once:", repeated)
print()
for r in orders:
    if r["order_id"] in repeated:
        print({k: r[k] for k in ("order_id", "segment", "amount", "status", "order_date")})

order_ids appearing more than once: ['KR4201']

{'order_id': 'KR4201', 'segment': 'Retail-Core', 'amount': '1865', 'status': 'returned', 'order_date': '2026-08-03'}
{'order_id': 'KR4201', 'segment': 'Retail-Core', 'amount': '1865', 'status': 'returned', 'order_date': '2026-09-14'}


Same order id. Same amount. Same status. Same segment. Six weeks apart.

So which is it?

The same order, exported twice, with the second export stamping the wrong date? Or a genuine repeat
order that reused an id it should not have?

The file cannot tell you and it never could.

In [11]:
kit.flow(["the dedupe that finds nothing", "the count that disagrees", "state an identity rule", "the order at the end of the column", "what ships"], lit=2)

## Section 3: an identity rule is something you state

Section 2 plus one new element: a rule, written down, that somebody else could apply.

"They look like duplicates" cannot be applied by anyone else. This can.

In [12]:
def find_duplicates(rows, key_fields):
    """Group rows by a stated identity rule and return the groups holding more than one row."""
    groups = {}
    for r in rows:
        key = tuple(r[f] for f in key_fields)
        groups.setdefault(key, []).append(r)
    return {k: v for k, v in groups.items() if len(v) > 1}

for rule in (["order_id"],
             ["order_id", "order_date"],
             ["order_id", "amount", "status"],
             list(orders[0].keys())):
    found = find_duplicates(orders, rule)
    rows_removed = sum(len(v) - 1 for v in found.values())
    print(f"{' + '.join(rule):58} groups {len(found)}, rows it would remove {rows_removed}")

order_id                                                   groups 1, rows it would remove 1
order_id + order_date                                      groups 0, rows it would remove 0
order_id + amount + status                                 groups 1, rows it would remove 1
order_id + customer_id + segment + amount + status + order_date + discount groups 0, rows it would remove 0


In [13]:
rules = [("order_id",), ("order_id", "order_date"), ("order_id", "amount"),
         tuple(orders[0].keys())]
kit.table(
    ["The identity rule", "Duplicate groups it finds"],
    [[" plus ".join(r) if len(r) < 4 else "every field", len(find_duplicates(orders, list(r)))]
     for r in rules],
    caption="Four defensible rules, three different answers",
)
kit.check("order_id alone finds the pair", len(find_duplicates(orders, ["order_id"])) == 1)
kit.check("order_id plus order_date finds nothing, because the dates differ",
          len(find_duplicates(orders, ["order_id", "order_date"])) == 0)
kit.check("the rule is the decision and the number follows from it",
          len(find_duplicates(orders, ["order_id"])) !=
          len(find_duplicates(orders, ["order_id", "order_date"])))

The identity rule,Duplicate groups it finds
order_id,1
order_id plus order_date,0
order_id plus amount,1
every field,0


### Who decides, drawn

In [14]:
kit.tree(
    {"label": "two rows share an order_id",
     "branches": [
         ("same order, entered twice", {"label": "keep one, and say which"}),
         ("two real orders, one id reused", {"label": "keep both, and fix the id upstream"}),
         ("you cannot tell", {"label": "keep both, flag the pair, escalate to the order book owner"})]},
    taken=["you cannot tell"], title="the rule is a business decision, not a code decision")

Four defensible rules, three different answers. The rule is the decision, and the number follows
from it rather than the other way round.

Note the last line: the whole-record rule finds nothing, which is where section 1 started.

### Who decides

Not you, on your own, on a Wednesday.

The pair goes to whoever owns the order book, with both rows on screen and your proposed rule
underneath. Until then the pair is flagged rather than deleted, and the flag is a decision too.

In [15]:
record_decision("order_id",
                f"{len(orders) - len(set(ids))} order_id shared by two rows ({repeated[0]})",
                "keep both rows, flag the pair",
                "the rows differ on order_date by six weeks, so re-export and repeat order are both "
                "plausible, and the order book owner decides which")

print(decisions[-1])

{'field': 'order_id', 'finding': '1 order_id shared by two rows (KR4201)', 'choice': 'keep both rows, flag the pair', 'reason': 'the rows differ on order_date by six weeks, so re-export and repeat order are both plausible, and the order book owner decides which'}


### Milestone: where this shows up in production

Every warehouse you will work in has a documented grain, meaning the statement of what one row
represents. "One row per order per export batch" and "one row per order" are different grains and
they disagree by exactly the pair you just found. Arguments about double-counted revenue are almost
always arguments about grain.

### Interview question this milestone just made answerable

"Two records share an id and disagree in one field. What do you do, and who decides?"

State an identity rule, flag rather than delete, and take it to whoever owns the data. The answer
that gets you rejected is deleting one of them because it looked like a duplicate.

In [16]:
kit.flow(["the dedupe that finds nothing", "the count that disagrees", "state an identity rule", "the order at the end of the column", "what ships"], lit=3)

## Section 4: the order at the end of the column

Section 3 plus one new element: the extremes.

Sort the column and read the tail. That is the entire technique, and it needs no arithmetic.

In [17]:
clean, rejects = clean_records(orders)
amounts = sorted(r["amount"] for r in clean)

print("smallest five:", amounts[:5])
print("largest five: ", amounts[-5:])

smallest five: [800, 950, 965, 970, 985]
largest five:  [2895, 2930, 2990, 2995, 480000]


In [18]:
kit.check("44 amounts convert and get sorted", len(amounts) == 44, f"{len(amounts)} amounts")
kit.check("the largest is Rs 480,000", max(amounts) == 480000, f"Rs {max(amounts):,}")
kit.check("it is more than 160 times the one below it",
          amounts[-1] / amounts[-2] > 160,
          f"Rs {amounts[-1]:,} against Rs {amounts[-2]:,}")

### The tail, and the decision at the end of it

In [19]:
kit.decision_ladder(["delete it, and the file lies about the business",
                     "keep it and say nothing, and the mean lies tomorrow",
                     "keep it, flag it, and say what it does to every summary"],
                    cut_at=2, title="what to do with an order that is real and enormous")

Four ordinary orders and then one that is over 160 times the one before it.

In [20]:
total = sum(amounts)
whale = amounts[-1]
print(f"orders that convert: {len(amounts)}")
print(f"total:               Rs {total}")
print(f"largest order:       Rs {whale}")
print(f"its share of the total: {round(100 * whale / total)} percent")
print(f"total without it:    Rs {total - whale}")

orders that convert: 44
total:               Rs 561145
largest order:       Rs 480000
its share of the total: 86 percent
total without it:    Rs 81145


One order out of fifty is 86 percent of the money in the file.

The instinct is to delete it, because it is ruining every number you might compute.

### Why the instinct is wrong

An outlier is a finding before it is a row. In a business where a typical order sits between Rs 800
and Rs 3,000, an order of Rs 480,000 is either the most interesting customer in this file or a data
entry error, and those two need opposite responses.

So look at it rather than at its size.

In [21]:
for r in clean:
    if r["amount"] == whale:
        print(r)

{'order_id': 'KR4232', 'customer_id': 'C1749', 'segment': 'Retail-Core', 'amount': 480000, 'status': 'delivered', 'order_date': '2026-08-19', 'discount': ''}


It converts cleanly. It has a customer, a date, a segment and a status like every other order.
Nothing about it is malformed. It is simply large.

So it survives cleaning, and it goes to tomorrow as a question rather than as a deletion.

In [22]:
record_decision("amount",
                f"one order at Rs {whale}, {round(100 * whale / total)} percent of the total",
                "keep, and raise with the order book owner",
                "it converts cleanly and is well formed, so it is real until somebody says otherwise")

# A simple fence, with no arithmetic beyond a multiple of the middle value.
middle = amounts[len(amounts) // 2]
fence = middle * 10
flagged = [a for a in amounts if a > fence]
print(f"middle order Rs {middle}, fence at ten times that is Rs {fence}")
print(f"orders above the fence: {flagged}")

middle order Rs 1955, fence at ten times that is Rs 19550
orders above the fence: [480000]


The fence is a convenience for spotting the tail quickly. It is not a test and it proves nothing.
The sorted tail on its own would have found the same order.

### Interview question

"The whale survived cleaning. Why?"

Because cleaning removes what cannot be read, and that order reads perfectly. Removing it would be
an analysis decision wearing a cleaning decision's clothes.

In [23]:
kit.flow(["the dedupe that finds nothing", "the count that disagrees", "state an identity rule", "the order at the end of the column", "what ships"], lit=4)

## Section 5: what ships

Section 4 plus one new element: the two files that leave this notebook.

The profiled dataset without the decisions log is an opinion.

In [24]:
FIELDS = list(orders[0].keys())

clean_path = f"{OUTPUT_DIR}/C2_W01_D03_profiled_orders_STUDENT.csv"
rejects_path = f"{OUTPUT_DIR}/C2_W01_D03_rejects_STUDENT.csv"
log_path = f"{OUTPUT_DIR}/C2_W01_D03_decisions_log_STUDENT.csv"

with open(clean_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS); w.writeheader(); w.writerows(clean)

with open(rejects_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["order_id", "reason"]); w.writeheader(); w.writerows(rejects)

with open(log_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["field", "finding", "choice", "reason"])
    w.writeheader(); w.writerows(decisions)

for p in (clean_path, rejects_path, log_path):
    print("wrote", p)

wrote output/C2_W01_D03_profiled_orders_STUDENT.csv
wrote output/C2_W01_D03_rejects_STUDENT.csv
wrote output/C2_W01_D03_decisions_log_STUDENT.csv


### The reconciliation, one level up

Yesterday it was input equals clean plus rejected. Today it also has to survive every decision you
made, including the pair you chose to keep.

In [25]:
with open(clean_path) as f:
    back_clean = list(csv.DictReader(f))
with open(rejects_path) as f:
    back_rejects = list(csv.DictReader(f))
with open(log_path) as f:
    back_log = list(csv.DictReader(f))

print(f"{len(orders)} in = {len(back_clean)} profiled + {len(back_rejects)} rejected")
assert len(back_clean) + len(back_rejects) == len(orders), "orders went missing"
print(f"decisions recorded: {len(back_log)}")
for d in back_log:
    print(f"  {d['field']:10} {d['choice']}")

50 in = 44 profiled + 6 rejected
decisions recorded: 2
  order_id   keep both rows, flag the pair
  amount     keep, and raise with the order book owner


In [26]:
kit.check("all three files reopened and parsed", len(back_clean) + len(back_rejects) > 0)
kit.check("the reconciliation holds across the files on disk",
          len(back_clean) + len(back_rejects) == len(orders),
          f"{len(back_clean)} clean plus {len(back_rejects)} rejected against {len(orders)} in")

Three files, read back from disk, and the counts add up. That is a defensible day's work.

## Section 6: the file that breaks the contract

One last thing arrives from the same upstream system: a companion export with the same fields.

In [27]:
with open(COMPANION_CSV) as f:
    companion = list(csv.DictReader(f))

print(f"rows read: {len(companion)}")
print("first row:", companion[0])

rows read: 21
first row: {'order_id': 'order_id', 'customer_id': 'customer_id', 'segment': 'segment', 'amount': 'amount', 'status': 'status', 'order_date': 'order_date', 'discount': 'discount'}


Read the first row again.

`DictReader` took its keys from line 1 and then handed you line 2 as data, and line 2 is another
copy of the header. So your first "order" has an `order_id` of `order_id`.

Nothing raised, because a header row is a perfectly valid row of text.

In [28]:
suspects = [r for r in companion if r["order_id"] == "order_id"]
print(f"rows that are actually a repeated header: {len(suspects)}")

usable = [r for r in companion if r["order_id"] != "order_id"]
print(f"rows after removing them: {len(usable)}")

record_decision("companion file",
                "line 2 repeats the header row",
                "drop rows whose order_id reads order_id, and tell the sender",
                "a repeated header is a export defect, not data, and it would become an order")
print(decisions[-1])

rows that are actually a repeated header: 1
rows after removing them: 20
{'field': 'companion file', 'finding': 'line 2 repeats the header row', 'choice': 'drop rows whose order_id reads order_id, and tell the sender', 'reason': 'a repeated header is a export defect, not data, and it would become an order'}


### What this notebook established

In [29]:
kit.table(
    ["The idea", "What proved it here"],
    [["A whole-record check finds only exact copies", "0 duplicates against 49 distinct ids on 50 rows"],
     ["Same record is a rule you state, not a fact the data holds", "four rules, three different answers"],
     ["An outlier is a finding before it is a row to delete", "Rs 480,000 is real and it is 86 percent of the money"],
     ["What ships is the data plus the decisions log", "three files, reopened, and the counts add up"]],
    caption="Day 3, notebook 2",
)
kit.flow(["the dedupe that finds nothing", "the count that disagrees", "state an identity rule", "the order at the end of the column", "what ships"], lit=4, title="the notebook, end to end")
kit.check_summary()

The idea,What proved it here
A whole-record check finds only exact copies,0 duplicates against 49 distinct ids on 50 rows
"Same record is a rule you state, not a fact the data holds","four rules, three different answers"
An outlier is a finding before it is a row to delete,"Rs 480,000 is real and it is 86 percent of the money"
What ships is the data plus the decisions log,"three files, reopened, and the counts add up"


This is the same lesson as the profile: the check that would have caught it is a comparison nobody
thought to make. `distinct` on `order_id` would have shown it too.

## Crux

A whole-record check finds only exact copies, so compare the id count against the row count every
time.

An identity rule is something you state, and whoever owns the data decides it.

An outlier is a finding to investigate before it is a row to delete.

The profiled dataset without its decisions log is an opinion.

## What tomorrow does with this

You have a dataset you can defend and a whale you decided to keep. Tomorrow you compute the average
order value and find out what that whale does to it.

Keep your three output files and your functions.